In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from __future__ import annotations

from typing import Callable, Iterable

from probeharbour.models import Candidate, Characterization
from probeharbour.rules import RuleSet
from probeharbour.specificity import SpecificityEvidence, SpecificityRequest, SpecificityService

In [ ]:
#| export
def evaluate_candidates(candidates: Iterable[Candidate], *, ruleset: RuleSet,
                        characterize: Callable[[Candidate], Characterization],
                        specificity: SpecificityService, database_id: str, database_version: str,
                        method_id: str) -> list[dict]:
    """Create selection-ready records without coupling to a thermo or BLAST backend."""
    records = []
    for candidate in candidates:
        thermo = characterize(candidate)
        explanation = ruleset.explain({"lig_dinuc": candidate.lig_dinuc, "tm_full": thermo.tm_full,
                                       "p3_homodimer_dg": thermo.p3_homodimer_dg,
                                       "has_simple_repeat": thermo.has_simple_repeat, "has_5g_repeat": thermo.has_5g_repeat})
        evidence, cache_hit = specificity.evaluate(SpecificityRequest(candidate.binder, candidate.transcript_id, database_id, database_version, method_id))
        records.append({"candidate_id": candidate.candidate_id, "candidate": candidate.as_dict(), "characterization": thermo.as_dict(),
                        "score": explanation.as_dict(), "recalculated_score": explanation.recalculated_score,
                        "specificity": evidence.as_dict(), "specificity_percent": evidence.specificity_percent, "specificity_cache_hit": cache_hit})
    return records